# 2. S1·S2 플랜 효과 비교

S1은 Action-level Distillation이고 S2는 Action+CoT Distillation입니다.
S2에서는 정상 플랜, 같은 태스크의 과거 플랜, 플랜 생략을 평가합니다.
S1은 Action-only 기준선으로 평가합니다.
모든 조건에서 같은 LIBERO+ 태스크·초기 상태·시드를 사용합니다.

1. BF16 CUDA GPU 런타임을 선택하고 Colab Secrets에 `HF_TOKEN`을 등록하세요.
2. `SOURCE_REF`에 이 평가 코드가 포함된 원격 Git 커밋을 넣으세요.
3. S1과 S2 체크포인트를 연결하세요. 동일한 초기 S0, 데이터, T1, 학습 설정, optimizer step을 사용해야 합니다.
   이미 등록한 HF asset 키는 설정 셀에서 `COT_CHECKPOINT`, `ACTION_CHECKPOINT`로 지정하세요.
   훈련 완료 로그에 출력된 S1과 S2의 `CHECKPOINT_PINS`를 하나의 딕셔너리로 합쳐 넣으세요.
   직접 입력하려면 키별로 `repo_id`, 불변 `revision` SHA, `prefix`를 지정하세요.
   `prefix` 형식은 `runs/<그룹>/S1/checkpoints/step-<7자리 번호>` 또는
   `runs/<그룹>/S2/checkpoints/step-<7자리 번호>`입니다.
4. `ACTION`을 `plan → setup → run` 순서로 바꾸며 설정 셀과 실행 셀을 실행하세요.
   먼저 `TRIALS=1`, `MAX_STEPS=32`로 파일럿을 하고, 본 평가에는 새 `RUN_NAME`을 사용하세요.

- **현재 vs 과거 플랜**은 현재 관측에 맞는 내용의 효과를 봅니다. 과거 조건은 각 에피소드에서 두 쿼리 전의 플랜을 사용합니다.
  처음 두 쿼리는 정상 플랜으로 시작합니다. **실제로 다른 플랜이 들어간 횟수**도 확인하세요.
- **현재 vs 플랜 생략**은 같은 체크포인트가 입력 플랜에 얼마나 의존하는지 봅니다.
- **현재 vs Action-only KD**는 최종 작업 성공률을 같은 초기 상태에서 비교합니다.

`paired_success.csv`의 성공률 차이와 두 조건 중 한쪽만 성공한 횟수를 확인하세요.
`action_probes.csv`는 정상 rollout 관측에서 **S0의 상태를 고정한 채** 플랜만 바꾼 정규화 액션 L1 차이입니다.
현재 플랜과 과거 플랜이 모두 유효하고 내용이 다른 쿼리만 이 파일에 포함합니다.
플랜을 바꿔 동작이 달라진 것과 성공률이 바뀐 것은 각각 해석하세요.
생성된 플랜이 비어 있으면 해당 쿼리에서 플랜 없이 실행하고 그 횟수를 별도로 남깁니다.


In [ ]:
from pathlib import Path
import os, re, subprocess, sys, time
os.environ.setdefault('VLADE_SESSION_STARTED_UNIX', str(time.time()))
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
sys.dont_write_bytecode = True
SOURCE_REF = 'refs/heads/main'  # 재개는 최초 실행의 SOURCE_COMMIT을 지정합니다.
REPOSITORY = 'https://github.com/eastha10/VLADE.git'
SOURCE_COMMIT = SOURCE_REF if re.fullmatch(r'[0-9a-f]{40}', SOURCE_REF) else subprocess.check_output(['git','ls-remote',REPOSITORY,SOURCE_REF], text=True).split()[0]
if not re.fullmatch(r'[0-9a-f]{40}', SOURCE_COMMIT):
    raise ValueError('불변 Git commit이 필요합니다.')
CODE = Path('/content/vlade_sources') / SOURCE_COMMIT
if any((name == 'vlade' or name.startswith('vlade.')) and getattr(module,'__file__',None) and not Path(module.__file__).resolve().is_relative_to(CODE) for name,module in list(sys.modules.items())):
    raise RuntimeError('다른 소스가 로드되어 있습니다. Colab 런타임을 다시 시작하십시오.')
if not CODE.exists():
    subprocess.run(['git','clone','--no-checkout','--filter=blob:none',REPOSITORY,str(CODE)],check=True)
    subprocess.run(['git','-C',str(CODE),'fetch','--depth','1','origin',SOURCE_COMMIT],check=True)
    subprocess.run(['git','-C',str(CODE),'checkout','--detach',SOURCE_COMMIT],check=True)
sys.path[:0] = [str(CODE/'src'),str(CODE)]
from vlade.utils.source_identity import source_identity
assert source_identity(CODE,require_clean=True)['git_commit']==SOURCE_COMMIT
print('SOURCE_COMMIT:',SOURCE_COMMIT)


In [ ]:
WORKSPACE = "/content/vlade_distillation"
RUN_NAME = "s1-s2-plan-effect-seed42"
ACTION = "plan"  #@param ["plan", "setup", "run"]
TRIALS = 10
MAX_STEPS = 600
OVERRIDES = dict(trials=TRIALS, max_steps=MAX_STEPS)

COT_CHECKPOINT = "hfasset://S2"
ACTION_CHECKPOINT = "hfasset://S1"
CHECKPOINT_PINS = {}
# 예: {"S2": {"repo_id": "VLADE-konkuk/VLADE-checkpoints",
#              "revision": "<40자리 HF commit>", "prefix": "runs/<그룹>/S2/checkpoints/step-0000100"}}
OVERRIDES.update(cot_checkpoint=COT_CHECKPOINT, action_checkpoint=ACTION_CHECKPOINT,
                 checkpoint_pins=CHECKPOINT_PINS)


In [ ]:
from vlade.research.plan_evaluation import job

out = job(CODE / 'configs/experiments/s1_s2_plan_effect.json', workspace=WORKSPACE,
          run_name=RUN_NAME, action=ACTION, overrides=OVERRIDES, colab=True)


In [ ]:
import json
from IPython.display import JSON, display
from vlade.research.plan_evaluation import output_path

out = output_path(Path(WORKSPACE), RUN_NAME)
if (out / "results.json").exists():
    display(JSON(json.loads((out / "results.json").read_text(encoding="utf-8"))))
else:
    print("결과 파일은 run 단계가 완료되면 생성됩니다.")
